# Phase 2.5: Price Level Features (Budget Input)

Extract price levels from tips and integrate as a feature for budget-aware recommendations.

This allows users to filter by budget: Low / Medium / High

In [2]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from config import *
from src.data.loader import load_businesses, load_tips
from src.data.cache import DataCache
from src.features.price_level_features import PriceLevelFeatureBuilder

cache = DataCache(str(CACHE_DIR))

# Load cached data from earlier phases
businesses = cache.load_dataframe('businesses_major_cities')
print(f"Loaded {len(businesses):,} businesses")

Loaded 10,341 businesses


## Load Tips Data

In [3]:
# Load tips (10% sample same as other data)
tips = load_tips(str(TIPS_FILE), sample=True, sample_fraction=PHASE1_SAMPLE_FRACTION)
print(f"Loaded {len(tips):,} tips")
print(f"\nTips columns: {tips.columns.tolist()}")
print(f"\nSample tips with price signals:")
print(tips[tips['text'].str.contains('price|cheap|expensive|deal', case=False, na=False)].head(3)[['business_id', 'text']])

Loaded 90,782 tips

Tips columns: ['user_id', 'business_id', 'text', 'date', 'compliment_count']

Sample tips with price signals:
               business_id                                               text
16  8Kw4-lW3-ag4s2s9ELQzgQ  Can't pay at the pump, but same price cash or ...
22  2uTT4-ZY21p_Py4pmbrbTg          Very fresh and tasty. Good deal at lunch.
23  Yk8DSN-nvFNJrWXizuzsIg                        Ok prices but slow service.


## Build Price Level Features

In [4]:
# Initialize builder
price_builder = PriceLevelFeatureBuilder()

# Build price levels for all businesses
print("Building price level features...")
price_features = price_builder.build_price_features(businesses, tips)

print(f"\n✓ Generated price levels for {len(price_features):,} businesses")
print(f"\nPrice level distribution:")
print(price_features['price_level'].value_counts().sort_index())
print(f"\nSource of price data:")
print(price_features['price_source'].value_counts())

Building price level features...
  Processed 100/10341 businesses
  Processed 200/10341 businesses
  Processed 300/10341 businesses
  Processed 400/10341 businesses
  Processed 500/10341 businesses
  Processed 600/10341 businesses
  Processed 700/10341 businesses
  Processed 800/10341 businesses
  Processed 900/10341 businesses
  Processed 1000/10341 businesses
  Processed 1100/10341 businesses
  Processed 1200/10341 businesses
  Processed 1300/10341 businesses
  Processed 1400/10341 businesses
  Processed 1500/10341 businesses
  Processed 1600/10341 businesses
  Processed 1700/10341 businesses
  Processed 1800/10341 businesses
  Processed 1900/10341 businesses
  Processed 2000/10341 businesses
  Processed 2100/10341 businesses
  Processed 2200/10341 businesses
  Processed 2300/10341 businesses
  Processed 2400/10341 businesses
  Processed 2500/10341 businesses
  Processed 2600/10341 businesses
  Processed 2700/10341 businesses
  Processed 2800/10341 businesses
  Processed 2900/10341 b

## Merge with Business Data

In [5]:
# Merge price levels back to businesses
businesses_with_price = businesses.merge(
    price_features[['business_id', 'price_level', 'price_source']], 
    on='business_id',
    how='left'
)

# Add human-readable labels
price_labels = {1: 'Low', 2: 'Medium', 3: 'High'}
businesses_with_price['price_label'] = businesses_with_price['price_level'].map(price_labels)

print(f"✓ Merged price levels with {len(businesses_with_price):,} businesses")
print(f"\nSample:")
print(businesses_with_price[['business_id', 'name', 'city', 'price_level', 'price_label', 'price_source']].head(10))

✓ Merged price levels with 10,341 businesses

Sample:
              business_id                              name            city  \
0  qkRM_2X51Yqxk3btlwAQIg                    Temple Beth-El  St. Petersburg   
1  bBDDEgkFA1Otx9Lfe7BZUQ                    Sonic Drive-In       Nashville   
2  il_Ro8jwPlHresjw9EGmBg                           Denny's    Indianapolis   
3  ROeacJQwBeh05Rqg7F6TCg                               BAP    Philadelphia   
4  noByYNtDLQAra9ccqxdfDw                               H&M   Santa Barbara   
5  tMkwHmWFUEXrC9ZduonpTg                The Green Pheasant       Nashville   
6  eYxGFkxo6m3SYGVTh5m2nQ  Big Boyz Toyz Motorcycle Rentals          Tucson   
7  oaboaRBUgGjbo2kfUIKDLQ                  Mike's Ice Cream       Nashville   
8  Q3kQYhkYxSRyYyeBgtk--A             Cook's Glass & Mirror    Indianapolis   
9  anLQj9AM8vjbcLSIE0iUgg                     Papa Murphy's          Tucson   

   price_level price_label price_source  
0            2      Medium      de

## Cache Price Features

In [6]:
# Save the enhanced business data
cache.save_dataframe(businesses_with_price, 'businesses_with_price')
cache.save_dataframe(price_features, 'price_levels')

print("✓ Price features cached")
print("\nReady for Phase 3: Feature Engineering with Budget")

✓ Price features cached

Ready for Phase 3: Feature Engineering with Budget


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Price level distribution
price_dist = businesses_with_price['price_label'].value_counts()
axes[0].bar(price_dist.index, price_dist.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[0].set_title('Price Level Distribution')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Price Level')

# Average stars by price level
avg_stars = businesses_with_price.groupby('price_label')['stars'].mean()
axes[1].bar(avg_stars.index, avg_stars.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Average Rating by Price Level')
axes[1].set_ylabel('Average Stars')
axes[1].set_xlabel('Price Level')
axes[1].set_ylim([0, 5])

plt.tight_layout()
plt.show()